# Nifty Volatility — v2, trained on real data

Forecasts next-day realized volatility of the Nifty 50 from price, **India VIX**
and news-sentiment features, then trades it three ways: a volatility-targeted
index overlay, a **long/short** directional book, and a variance-risk-premium
carry strategy.

This is a rebuild after a critical review of a first version whose backtest lost
to buy-and-hold. That version's three bugs, all fixed here:

| Bug | Effect |
|---|---|
| Sized positions off a **Garman-Klass** forecast (intraday vol) while trading close-to-close returns (includes the overnight gap) | every position ~1.24x too large |
| `exp(mean of log vol)` is the **median**, not the mean | a further ~1.09x |
| Rebalanced daily off a noisy daily forecast, and charged no financing on leverage | ~3.5%/yr of cost drag |

Predicted over-leverage 1.35x; leverage backed out of the original results 1.31x.

---

### Kaggle setup — do this first

In the right-hand panel:

1. **Settings → Internet → On.** Required for yfinance, GDELT and HuggingFace.
   Kaggle asks for phone verification to enable it.
2. **Settings → Accelerator → GPU T4 x2.** Only needed for the FinBERT step;
   skip it if you set `RUN_NEWS = False` below.

Runtime is roughly 5 minutes without news, 25–40 with the full GDELT backfill.
Intermediate results are cached to `/kaggle/working` so a re-run is fast.

## 0. Setup

In [ ]:
!pip install -q yfinance arch optuna shap feedparser lightgbm --upgrade 2>&1 | tail -2

In [ ]:
import os, re, time, json, pathlib, warnings
warnings.filterwarnings("ignore")

from datetime import datetime, timedelta
from dataclasses import dataclass, field
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import requests
import feedparser
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta

try:
    import pytz
    IST = pytz.timezone("Asia/Kolkata")
except ImportError:
    from zoneinfo import ZoneInfo
    IST = ZoneInfo("Asia/Kolkata")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORK = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path(".")
print("output dir:", WORK)

## 1. Config

In [ ]:
# ---- data -----------------------------------------------------------------
TICKER          = "^NSEI"
VIX_TICKER      = "^INDIAVIX"
PRICE_YEARS     = 10

# ---- news -----------------------------------------------------------------
RUN_NEWS        = True     # False -> price + VIX only (fast, still a full run)
GDELT_CHUNK_DAYS = 7       # smaller = more coverage, more requests (250/response cap)
GDELT_START_MIN = "2017-01-01"   # the DOC API does not index reliably before this

# ---- market assumptions ---------------------------------------------------
RF_ANNUAL       = 0.065    # ~MIBOR; financing and the excess-return Sharpe
DIV_YIELD       = 0.012    # ^NSEI is a PRICE index -- dividends are not in it
COST_BPS_CASH   = 7.5
COST_BPS_FUT    = 5.0      # Nifty futures: 2 bps STT on the sell side + spread
TARGET_ANN_VOL  = 0.12

# ---- model ----------------------------------------------------------------
N_TRIALS        = 25
TRAIN_YEARS     = 3
TEST_MONTHS     = 2
RANDOM_SEED     = 42

RELEVANCE_KEYWORDS = [
    "nifty", "sensex", "rbi", "reserve bank of india", "budget", "inflation",
    "fii", "dii", "foreign institutional investor", "domestic institutional investor",
    "sebi", "repo rate", "gdp india", "indian rupee", "bse", "nse india",
    "reliance industries", "hdfc bank", "icici bank", "infosys", "tcs",
    "tata consultancy", "kotak mahindra", "larsen", "itc limited", "axis bank",
    "state bank of india", "bharti airtel", "bajaj finance", "hindustan unilever",
    "maruti suzuki", "sun pharma", "asian paints", "adani",
]
GDELT_QUERY = '(Nifty OR Sensex OR "RBI" OR "Union Budget" OR "Indian rupee" OR FII OR DII)'
RSS_FEEDS = {
    "moneycontrol": "https://www.moneycontrol.com/rss/marketreports.xml",
    "economic_times": "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
    "livemint": "https://www.livemint.com/rss/markets",
    "business_standard": "https://www.business-standard.com/rss/markets-106.rss",
    "google_news_nifty": "https://news.google.com/rss/search?q=Nifty+when:2d&hl=en-IN&gl=IN&ceid=IN:en",
}

## 2. Volatility estimators — the unit conventions everything depends on

Two different daily volatilities, and v1 conflated them:

- **Intraday** (open-to-close) — what Garman-Klass measures.
- **Total** (close-to-close) — what you actually trade, gap included.

Everything downstream works in total units, which is also what India VIX quotes
in. That is what makes the two comparable and the VRP strategy possible.

In [ ]:
# ===== src/volatility.py =====
"""Volatility estimators, and the unit conventions the rest of the pipeline depends on.

The single most important idea in this module is that there are *two different*
daily volatilities and v1 of this project conflated them:

  * **Intraday (open-to-close) vol** -- what Garman-Klass measures. It is built
    from that session's O/H/L/C and knows nothing about the gap between
    yesterday's close and today's open.
  * **Total (close-to-close) vol** -- what you actually trade. A position held
    overnight earns ``log(C_t / C_{t-1})``, whose variance includes the
    overnight gap.

For the Nifty the overnight gap is a large minority of total daily variance
(roughly a third), so ``sigma_GK`` systematically understates the risk of a
close-to-close position by ~20%. Sizing a position as
``target_vol / predicted_GK_vol`` therefore over-levers by ~1/0.8 = 1.25x
*before* anything else goes wrong. See CRITICAL_ANALYSIS.md section 1.

Everything downstream of this module works in **total** vol units, which are
also the units India VIX quotes in -- that is what makes the two comparable
and the variance-risk-premium strategy possible at all.
"""


import numpy as np
import pandas as pd

TRADING_DAYS = 252
_GK_CO_COEF = 2.0 * np.log(2.0) - 1.0


def garman_klass_variance(df: pd.DataFrame) -> pd.Series:
    """Garman-Klass *intraday* variance from one session's own OHLC.

        sigma^2_GK = 0.5 * ln(H/L)^2 - (2 ln2 - 1) * ln(C/O)^2

    Floored at zero: the estimator is unbiased but not guaranteed positive on
    any single observation.
    """
    log_hl = np.log(df["high"] / df["low"])
    log_co = np.log(df["close"] / df["open"])
    return (0.5 * log_hl**2 - _GK_CO_COEF * log_co**2).clip(lower=0.0)


def gap_variance(df: pd.DataFrame) -> pd.Series:
    """Overnight variance contribution, ``ln(O_t / C_{t-1})^2``.

    A one-observation estimator of the overnight component, exactly analogous
    to using ``r_t^2`` as a one-observation estimator of daily variance: noisy
    per day, unbiased in expectation, which is all we need since it is being
    summed into a target that gets modelled in logs.
    """
    return np.log(df["open"] / df["close"].shift(1)) ** 2


def total_daily_variance(df: pd.DataFrame) -> pd.Series:
    """Gap-inclusive daily variance = overnight variance + intraday variance.

    This is the decomposition of a close-to-close return into its two legs. It
    is the correct target for anything that sizes or prices an *overnight*
    position, and it is directly comparable to India VIX (de-annualized).
    """
    return gap_variance(df) + garman_klass_variance(df)


def realized_vol(df: pd.DataFrame, kind: str = "total") -> pd.Series:
    """Daily volatility (not variance, not annualized) for the given estimator."""
    if kind == "total":
        var = total_daily_variance(df)
    elif kind == "gk":
        var = garman_klass_variance(df)
    elif kind == "close":
        var = np.log(df["close"] / df["close"].shift(1)) ** 2
    else:
        raise ValueError(f"unknown estimator kind: {kind!r}")
    return np.sqrt(var)


def annualize(daily_vol: pd.Series | float, periods: int = TRADING_DAYS):
    return daily_vol * np.sqrt(periods)


def deannualize(annual_vol: pd.Series | float, periods: int = TRADING_DAYS):
    return annual_vol / np.sqrt(periods)


# --------------------------------------------------------------------------
# log-space -> level-space conversion
# --------------------------------------------------------------------------

def smearing_factor(log_residuals: np.ndarray | pd.Series) -> float:
    """Duan's smearing estimate of ``E[exp(residual)]``.

    Models are fit on ``log`` volatility because volatility is right-skewed and
    strictly positive. But ``exp(E[log v])`` is the *median*, not the mean --
    it understates the level by roughly ``exp(sigma_resid^2 / 2)``. With the
    residual spread this model actually achieves (~0.41 in log units) that is
    an 8-9% low bias, which feeds straight into over-leverage when you divide
    by it.

    Duan's smearing estimator is the non-parametric version of that correction
    and does not assume the residuals are Gaussian. It MUST be estimated on
    training residuals only.
    """
    resid = np.asarray(log_residuals, dtype=float)
    resid = resid[np.isfinite(resid)]
    if resid.size == 0:
        return 1.0
    return float(np.mean(np.exp(resid)))


def log_to_level(log_pred: pd.Series | np.ndarray, smearing: float = 1.0):
    """Convert a log-vol forecast to a vol level, applying the mean correction."""
    return np.exp(log_pred) * smearing

## 3. Price data and India VIX

India VIX was absent from v1 entirely and is the biggest single omission: it is
a forward-looking, market-consensus volatility forecast, published daily and
free, and it already impounds most of what news sentiment could carry.

In [ ]:
import yfinance as yf


def load_yf(ticker, years):
    df = yf.download(ticker, period=f"{years}y", interval="1d",
                     auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.rename(columns=str.lower)
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "date"
    return df


prices = load_yf(TICKER, PRICE_YEARS)[["open", "high", "low", "close", "volume"]].dropna()
vix = load_yf(VIX_TICKER, PRICE_YEARS)["close"].dropna().rename("vix")

assert len(prices) > 500, "price download failed -- is Internet enabled in Settings?"
print(f"{TICKER}: {len(prices)} sessions  {prices.index.min().date()} -> {prices.index.max().date()}")
print(f"{VIX_TICKER}: {len(vix)} sessions  mean {vix.mean():.1f}  max {vix.max():.1f}")
prices.tail(3)

In [ ]:
# The v1 bug, measured on your own data rather than assumed.
gk = realized_vol(prices, "gk")
total = realized_vol(prices, "total")
share = (gk / total).median()
cc_vol = np.log(prices["close"] / prices["close"].shift(1)).std() * np.sqrt(252)

print(f"Nifty annualized close-to-close vol : {cc_vol:.1%}")
print(f"median sigma_GK / sigma_total       : {share:.3f}")
print(f"-> sizing off Garman-Klass over-levers by {1/share:.2f}x")
print(f"overnight share of daily variance   : {1 - (gk**2).mean()/(total**2).mean():.1%}")

## 4. News: GDELT backfill, dedupe, relevance filter, FinBERT

Carried over from v1 and unchanged in substance. The only thing the rest of the
pipeline needs out of this section is a frame with `trading_day`, `polarity`,
`positive`, `negative`.

Bucketing rule: a story belongs to the trading day whose **pre-open window**
(previous close 15:30 IST → open 09:15 IST) contains it. Nothing is thrown
away — intraday news rolls into the next session, which is the first moment you
could have traded on it.

In [ ]:
# ===== news collection: GDELT backfill + live RSS =====
GDELT_DOC_ENDPOINT = "https://api.gdeltproject.org/api/v2/doc/doc"


def _query_gdelt_window(query, start, end, max_records=250,
                        source_country=None, timeout=30):
    full = f"{query} sourcecountry:{source_country}" if source_country else query
    params = {"query": full, "mode": "artlist", "maxrecords": max_records,
              "format": "json", "sort": "datedesc",
              "startdatetime": start.strftime("%Y%m%d%H%M%S"),
              "enddatetime": end.strftime("%Y%m%d%H%M%S")}
    try:
        r = requests.get(GDELT_DOC_ENDPOINT, params=params, timeout=timeout,
                         headers={"User-Agent": "Mozilla/5.0 (research; nifty-vol)"})
        r.raise_for_status()
        return r.json().get("articles", [])
    except (requests.RequestException, ValueError):
        return []


def collect_gdelt_history(query, start_date, end_date, source_country="India",
                          chunk_days=7, sleep_sec=1.0, max_records=250,
                          progress_every=20):
    """Walk the date range in windows, one DOC API request each.

    GDELT caps a single response at 250 articles, so a shorter `chunk_days`
    buys more coverage at the cost of more requests. The DOC API only indexes
    back to roughly 2017 -- earlier windows come back empty rather than error.
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    rows, cursor, n = [], start, 0
    total = max(1, (end - start).days // chunk_days)

    while cursor < end:
        window_end = min(cursor + timedelta(days=chunk_days), end)
        for art in _query_gdelt_window(query, cursor, window_end,
                                       max_records, source_country):
            try:
                ts = datetime.strptime(art.get("seendate"), "%Y%m%dT%H%M%SZ")
            except (TypeError, ValueError):
                continue
            rows.append({"headline": (art.get("title") or "").strip(),
                         "source": art.get("domain", "gdelt"),
                         "published_at": ts, "url": art.get("url", "")})
        cursor = window_end
        n += 1
        if n % progress_every == 0:
            print(f"    GDELT window {n}/{total}  ({len(rows)} articles so far)")
        time.sleep(sleep_sec)

    df = pd.DataFrame(rows, columns=["headline", "source", "published_at", "url"])
    df = df[df["headline"].str.len() > 0]
    return df.drop_duplicates(subset=["url"]).reset_index(drop=True)


def collect_rss_news(feeds):
    frames = []
    for name, url in feeds.items():
        try:
            parsed = feedparser.parse(url, request_headers={"User-Agent": "Mozilla/5.0"})
        except Exception:
            continue
        rows = []
        for e in parsed.entries:
            h = getattr(e, "title", "").strip()
            if not h:
                continue
            t = None
            for key in ("published_parsed", "updated_parsed"):
                v = getattr(e, key, None)
                if v is not None:
                    t = datetime(*v[:6])
                    break
            rows.append({"headline": h, "source": name,
                         "published_at": t or datetime.utcnow(),
                         "url": getattr(e, "link", "")})
        if rows:
            frames.append(pd.DataFrame(rows))
    if not frames:
        return pd.DataFrame(columns=["headline", "source", "published_at", "url"])
    return pd.concat(frames, ignore_index=True).drop_duplicates(subset=["url"]).reset_index(drop=True)


# ===== preprocessing =====
_WS_PUNCT_RE = re.compile(r"[^a-z0-9\s]")


def normalize_headline(text):
    text = _WS_PUNCT_RE.sub(" ", str(text).lower().strip())
    return re.sub(r"\s+", " ", text).strip()


def dedupe_headlines(df, fuzzy_threshold=0.9):
    """Exact dedupe on normalised text, then a fuzzy pass within each day.

    Wire copy gets republished across outlets, so without this an article
    count mostly measures syndication rather than distinct news. The fuzzy
    pass is restricted to same-day buckets because near-duplicates of a story
    appear on the same day, and an all-pairs comparison would be O(n^2).
    """
    if df.empty:
        return df
    out = df.copy()
    out["_norm"] = out["headline"].map(normalize_headline)
    out = out.drop_duplicates(subset=["_norm"])
    out["_date"] = pd.to_datetime(out["published_at"]).dt.date

    keep = pd.Series(True, index=out.index)
    for _, grp in out.groupby("_date"):
        seen = []
        for idx, norm in grp["_norm"].items():
            if any(SequenceMatcher(None, norm, s).ratio() >= fuzzy_threshold for s in seen):
                keep.loc[idx] = False
            else:
                seen.append(norm)
    return out[keep].drop(columns=["_norm", "_date"]).reset_index(drop=True)


def filter_relevant(df, keywords):
    if df.empty:
        return df
    pat = re.compile(r"\b(" + "|".join(re.escape(k.lower()) for k in keywords) + r")\b")
    mask = df["headline"].str.lower().apply(lambda h: bool(pat.search(h)))
    return df[mask].reset_index(drop=True)


def bucket_news_to_trading_days(news_df, trading_days):
    """Assign each item to the trading day whose pre-open window contains it.

    A story published at `ts` belongs to the earliest trading day `D` whose
    09:15 IST open is at or after `ts` -- equivalently the window
    (D-1 close 15:30, D open 09:15]. Stated that way it handles weekends and
    holidays automatically, because it only ever matches real trading days.
    Nothing is discarded: intraday news simply rolls into the next session,
    which is the first point at which you could have traded on it.
    """
    if news_df.empty:
        out = news_df.copy()
        out["trading_day"] = pd.Series(dtype="datetime64[ns]")
        return out

    out = news_df.copy()
    ts = pd.to_datetime(out["published_at"])
    ts = ts.dt.tz_localize("UTC") if ts.dt.tz is None else ts
    ts_ist = ts.dt.tz_convert(IST)

    days = pd.DatetimeIndex(sorted(pd.to_datetime(trading_days)))
    opens = (days + pd.Timedelta(hours=9, minutes=15)).tz_localize(IST)

    pos = np.searchsorted(opens.values, ts_ist.values, side="left")
    valid = pos < len(days)
    out = out.loc[valid].copy()
    out["trading_day"] = days[pos[valid]]
    return out.reset_index(drop=True)


# ===== FinBERT =====
class FinBertScorer:
    """ProsusAI/finbert, inference only. polarity = P(pos) - P(neg)."""

    def __init__(self, model_name="ProsusAI/finbert", device=None, batch_size=64):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model = self.model.to(self.device).eval()
        self.id2label = self.model.config.id2label

    def score_all(self, headlines):
        torch = self.torch
        texts = headlines.fillna("").astype(str).tolist()
        labels = [self.id2label[i].lower() for i in range(len(self.id2label))]

        chunks = []
        with torch.no_grad():
            for i in range(0, len(texts), self.batch_size):
                batch = texts[i:i + self.batch_size]
                enc = self.tokenizer(batch, padding=True, truncation=True,
                                     max_length=64, return_tensors="pt").to(self.device)
                probs = torch.softmax(self.model(**enc).logits, dim=-1)
                chunks.append(probs.cpu().numpy())
                if (i // self.batch_size) % 200 == 0 and i:
                    print(f"    FinBERT {i}/{len(texts)}")
        arr = np.concatenate(chunks) if chunks else np.zeros((0, len(labels)))
        out = pd.DataFrame(arr, columns=labels)
        out["polarity"] = out["positive"] - out["negative"]
        return out


def score_headlines(df, scorer=None, batch_size=64):
    scorer = scorer or FinBertScorer(batch_size=batch_size)
    scores = scorer.score_all(df["headline"])
    out = df.reset_index(drop=True).copy()
    for c in ("positive", "negative", "neutral", "polarity"):
        out[c] = scores[c].to_numpy()
    return out

In [ ]:
NEWS_CACHE = WORK / "scored_news.parquet"
scored_news = None

if RUN_NEWS and NEWS_CACHE.exists():
    scored_news = pd.read_parquet(NEWS_CACHE)
    print(f"loaded cached news: {len(scored_news)} scored headlines")

elif RUN_NEWS:
    start = max(prices.index.min().strftime("%Y-%m-%d"), GDELT_START_MIN)
    end = prices.index.max().strftime("%Y-%m-%d")
    print(f"GDELT backfill {start} -> {end} (this is the slow part)")

    gdelt = collect_gdelt_history(GDELT_QUERY, start, end,
                                  chunk_days=GDELT_CHUNK_DAYS)
    rss = collect_rss_news(RSS_FEEDS)
    raw = pd.concat([gdelt, rss], ignore_index=True)
    print(f"  raw: {len(raw)}  (gdelt {len(gdelt)}, rss {len(rss)})")

    if len(raw):
        deduped = dedupe_headlines(raw)
        relevant = filter_relevant(deduped, RELEVANCE_KEYWORDS)
        bucketed = bucket_news_to_trading_days(relevant, prices.index)
        print(f"  deduped {len(deduped)} -> relevant {len(relevant)} -> bucketed {len(bucketed)}")

        if len(bucketed):
            scored_news = score_headlines(bucketed, batch_size=64)
            scored_news.to_parquet(NEWS_CACHE)
            print(f"  scored {len(scored_news)}; cached to {NEWS_CACHE}")

if scored_news is not None and len(scored_news):
    per_day = scored_news.groupby("trading_day").size()
    print(f"\ncoverage: {len(per_day)} trading days, median {per_day.median():.0f} headlines/day")
    print(f"date range: {per_day.index.min().date()} -> {per_day.index.max().date()}")
    display(scored_news[["headline", "polarity"]].tail(5))
else:
    print("\nRunning price + VIX only.")

## 5. Features

The sentiment block is rebuilt around what actually moves volatility. v1 fed a
*signed* mean polarity to predict a *magnitude* — but good news and bad news
both raise volatility, so the sign is nearly irrelevant, and averaging ~50
headlines crushes the remaining variance. What matters is intensity, disagreement
and news-volume shocks. Missing days stay NaN rather than being filled with 0.0,
which in v1 made "no news" indistinguishable from "perfectly balanced news".

The signed polarity is not wasted — it reappears in section 9 as a *directional*
signal, which is the question its sign is actually about.

In [ ]:
# ===== src/features.py =====
"""Feature construction.

Three changes from v1, in descending order of how much they matter:

1. **India VIX is now a feature.** Its absence was the single biggest miss in
   v1. Implied volatility is a forward-looking, market-consensus volatility
   forecast; it is the strongest known predictor of next-day realized vol and
   it subsumes most of what news sentiment could ever tell you. A vol model
   that does not use it is competing with one hand tied.

2. **Targets are gap-inclusive and multi-horizon.** See src/volatility.py for
   why. ``h=1`` keeps the error metrics comparable with v1; ``h=5`` is what the
   tradeable weekly-expiry strategy actually needs.

3. **Sentiment features measure intensity and surprise, not direction.**
   v1 fed the model ``mean(polarity)`` -- a *signed* quantity -- and asked it to
   predict a *magnitude*. Good news and bad news both raise volatility, so the
   sign is close to irrelevant and averaging ~50 headlines crushes what little
   variance remains. What actually moves volatility is how much is being
   written, how extreme it is, and how much the outlets disagree. Missing days
   are also left as NaN rather than filled with 0.0: in v1, "no news" and
   "perfectly balanced news" were encoded identically, which is a contaminated
   feature. LightGBM routes NaN natively.
"""


import numpy as np
import pandas as pd


EPS = 1e-8

PRICE_FEATURES = [
    "log_rv", "log_rv_w", "log_rv_m", "log_rv_q",
    "log_gk", "log_gap_share",
    "ret_1d", "ret_5d", "neg_ret_1d",
    "rv_trend", "rv_of_rv",
    "day_of_week",
]

VIX_FEATURES = [
    "log_iv", "log_vrp", "vix_chg_1d", "vix_chg_5d", "log_iv_slope",
]

SENTIMENT_FEATURES = [
    "sent_abs_mean", "sent_disp", "sent_tail_share", "sent_neg_share",
    "sent_skew", "news_vol_z", "sent_abs_shock", "sent_disp_roll3",
    "sent_abs_mean_roll5", "has_news",
]


# --------------------------------------------------------------------------
# price / autoregressive block
# --------------------------------------------------------------------------

def build_price_features(prices: pd.DataFrame) -> pd.DataFrame:
    """Realized-vol, HAR-style and return features, all known at date t's close."""
    out = prices.copy()

    out["log_return"] = np.log(out["close"] / out["close"].shift(1))
    out["rv"] = realized_vol(out, "total")      # gap-inclusive -- what we trade
    out["gk_vol"] = realized_vol(out, "gk")     # intraday only -- kept for reference
    out["log_rv"] = np.log(out["rv"] + EPS)
    out["log_gk"] = np.log(out["gk_vol"] + EPS)

    # Share of the day's variance that came from the overnight gap. Regime
    # information: gap-heavy periods are event-driven, range-heavy are grind.
    gk_var = garman_klass_variance(out)
    out["log_gap_share"] = np.log(
        ((out["rv"] ** 2 - gk_var).clip(lower=0) + EPS) / (out["rv"] ** 2 + EPS)
    )

    # HAR (Corsi 2009) cascade: daily / weekly / monthly / quarterly log-RV.
    out["log_rv_w"] = out["log_rv"].rolling(5).mean()
    out["log_rv_m"] = out["log_rv"].rolling(22).mean()
    out["log_rv_q"] = out["log_rv"].rolling(66).mean()

    out["ret_1d"] = out["log_return"]
    out["ret_5d"] = out["log_return"].rolling(5).sum()
    # Leverage effect: downside returns raise future vol far more than upside.
    out["neg_ret_1d"] = out["log_return"].clip(upper=0.0)

    out["rv_trend"] = out["log_rv_w"] - out["log_rv_m"]
    out["rv_of_rv"] = out["log_rv"].rolling(22).std()

    out["day_of_week"] = out.index.dayofweek
    return out


def add_vix_features(df: pd.DataFrame, vix_close: pd.Series | None) -> pd.DataFrame:
    """Join India VIX and derive the implied-vs-realized spread.

    ``vix_close`` is the India VIX closing level in annualized percent (the way
    it is quoted). The close at date t is known at t's close, so using it to
    forecast t+1 is not leakage.
    """
    out = df.copy()
    if vix_close is None or len(vix_close) == 0:
        for col in VIX_FEATURES:
            out[col] = np.nan
        out["iv_daily"] = np.nan
        return out

    vix = vix_close.reindex(out.index).ffill(limit=3)
    out["vix"] = vix
    out["iv_daily"] = deannualize(vix / 100.0)
    out["log_iv"] = np.log(out["iv_daily"] + EPS)

    # The variance risk premium in log space -- implied vol richness over
    # trailing realized. Strongly mean-reverting, and the core trading signal.
    out["log_vrp"] = out["log_iv"] - out["log_rv_w"]

    out["vix_chg_1d"] = out["log_iv"].diff()
    out["vix_chg_5d"] = out["log_iv"].diff(5)
    # Term-structure proxy: spot IV vs its own recent average.
    out["log_iv_slope"] = out["log_iv"] - out["log_iv"].rolling(22).mean()
    return out


# --------------------------------------------------------------------------
# sentiment block
# --------------------------------------------------------------------------

def aggregate_daily_sentiment(scored_news: pd.DataFrame) -> pd.DataFrame:
    """One row per trading day of *volatility-relevant* sentiment aggregates.

    Expects the columns produced by the v1 FinBERT step: ``trading_day``,
    ``polarity`` (= P(pos) - P(neg)), ``positive``, ``negative``.
    """
    cols = ["trading_day", "sent_mean", "sent_abs_mean", "sent_disp",
            "sent_tail_share", "sent_neg_share", "sent_skew", "article_count"]
    if scored_news is None or len(scored_news) == 0:
        return pd.DataFrame(columns=cols)

    news = scored_news.copy()
    news["abs_polarity"] = news["polarity"].abs()
    # "Tail" = an unambiguously charged headline. Averaging polarity over a day
    # hides these; their *count* is what tracks volatility.
    news["is_tail"] = (news["abs_polarity"] > 0.8).astype(float)
    news["is_negative"] = (news["negative"] > news["positive"]).astype(float)

    g = news.groupby("trading_day")
    daily = g.agg(
        sent_mean=("polarity", "mean"),
        sent_abs_mean=("abs_polarity", "mean"),
        sent_disp=("polarity", "std"),
        sent_tail_share=("is_tail", "mean"),
        sent_neg_share=("is_negative", "mean"),
        sent_skew=("polarity", "skew"),
        article_count=("polarity", "size"),
    ).reset_index()

    # std/skew are undefined for a 1-headline day; 0 dispersion is the honest
    # reading, but skew is genuinely unknown.
    daily["sent_disp"] = daily["sent_disp"].fillna(0.0)
    return daily[cols]


def add_sentiment_dynamics(daily: pd.DataFrame, baseline_window: int = 60) -> pd.DataFrame:
    """Turn level features into *stationary* surprise features.

    Raw ``article_count`` drifts with outlet coverage and with GDELT's own
    indexing over the years, so a tree that learns "count > 80 means high vol"
    in fold 3 is learning a date proxy, not a signal. What is stationary is the
    count relative to its own recent baseline -- a news-volume shock.
    """
    out = daily.sort_values("trading_day").reset_index(drop=True)

    log_count = np.log1p(out["article_count"])
    roll_mu = log_count.rolling(baseline_window, min_periods=10).mean()
    roll_sd = log_count.rolling(baseline_window, min_periods=10).std()
    out["news_vol_z"] = (log_count - roll_mu) / roll_sd.replace(0, np.nan)

    abs_mu = out["sent_abs_mean"].rolling(baseline_window, min_periods=10).mean()
    abs_sd = out["sent_abs_mean"].rolling(baseline_window, min_periods=10).std()
    out["sent_abs_shock"] = (out["sent_abs_mean"] - abs_mu) / abs_sd.replace(0, np.nan)

    out["sent_disp_roll3"] = out["sent_disp"].rolling(3, min_periods=1).mean()
    out["sent_abs_mean_roll5"] = out["sent_abs_mean"].rolling(5, min_periods=1).mean()
    return out


def build_feature_table(prices: pd.DataFrame,
                        scored_news: pd.DataFrame | None = None,
                        vix_close: pd.Series | None = None,
                        horizons: tuple[int, ...] = (1, 5),
                        baseline_window: int = 60) -> pd.DataFrame:
    """Assemble the full daily feature table plus targets for each horizon."""
    table = build_price_features(prices)
    table = add_vix_features(table, vix_close)

    daily = aggregate_daily_sentiment(scored_news)
    if len(daily) > 0:
        daily = add_sentiment_dynamics(daily, baseline_window)
        table = table.merge(
            daily.set_index("trading_day"),
            left_index=True, right_index=True, how="left",
        )
        table["has_news"] = table["article_count"].notna().astype(float)
        # Deliberately NOT filling the sentiment columns: NaN means "no news",
        # which is different information from "neutral news".
    else:
        for col in SENTIMENT_FEATURES:
            table[col] = np.nan
        table["has_news"] = 0.0

    return add_targets(table, horizons)


def add_targets(table: pd.DataFrame, horizons: tuple[int, ...] = (1, 5)) -> pd.DataFrame:
    """Forward realized volatility over the next ``h`` trading days.

    Two targets per horizon, because they are used for different things and
    conflating them is a real source of error:

    ``target_log_rv_h{h}``
        Mean of forward *log*  This is the modelling target -- log-vol is
        near-Gaussian and homoscedastic, which is what a squared-error
        objective wants.

    ``target_rvar_h{h}``
        Mean of forward *variance*, i.e. arithmetic realized variance. This is
        what a variance swap actually settles against, and it is strictly
        larger than the squared geometric mean by Jensen. Using the log-mean
        target to settle the VRP payoff understates realized variance on
        exactly the spiky days that hurt a short-vol book, which flatters the
        strategy precisely where it should be punished.
    """
    out = table.copy()
    var = out["rv"] ** 2
    for h in horizons:
        out[f"target_log_rv_h{h}"] = (
            out["log_rv"].shift(-1).rolling(h, min_periods=h).mean().shift(-(h - 1)))
        out[f"target_rvar_h{h}"] = (
            var.shift(-1).rolling(h, min_periods=h).mean().shift(-(h - 1)))
    return out


def feature_columns(table: pd.DataFrame, use_sentiment: bool = True,
                    use_vix: bool = True) -> list[str]:
    cols = [c for c in PRICE_FEATURES if c in table.columns]
    if use_vix:
        cols += [c for c in VIX_FEATURES if c in table.columns
                 and table[c].notna().any()]
    if use_sentiment:
        cols += [c for c in SENTIMENT_FEATURES if c in table.columns
                 and table[c].notna().any()]
    return cols

In [ ]:
table = build_feature_table(prices, scored_news, vix, horizons=(1, 5))
cols = feature_columns(table)
print(f"{len(cols)} features, {len(table)} rows\n")
print(cols)
table[["close", "rv", "gk_vol", "iv_daily", "log_vrp", "target_log_rv_h1"]].tail()

## 6. Baselines

**HAR-RV** (Corsi 2009) is the bar that matters. v1 compared against naive
persistence — `log(sigma_t)` from a single day's range, a very noisy level
estimate — and beating it by 15% is close to free.

GARCH is also repaired: v1 fit close-to-close returns (forecasting *total* vol)
and scored the result against a Garman-Klass *intraday* target, a near-constant
log offset of about +0.21. Most of v1's "+30.8% vs GARCH" was that unit error.

In [ ]:
# ===== src/baselines.py =====
"""Baselines the model has to beat -- and the reason v1's margins were inflated.

v1 reported "+15.0% RMSE vs naive" and "+30.8% vs GARCH". Both numbers are
softer than they look:

* **Naive persistence** is ``log(sigma_t)`` from a *single day's* range. As an
  estimator of the current vol level it is extremely noisy, so beating it by
  15% is close to free -- a 5-day moving average of the same quantity does most
  of it. It is a sanity check, not a competitor.

* **GARCH was scored in the wrong units.** ``fit_garch_forecast`` fits
  close-to-close returns, so it forecasts *total* (gap-inclusive) vol, but it
  was scored against a Garman-Klass *intraday* target. On a log scale that is a
  near-constant offset of ``log(sigma_total / sigma_GK) ~ +0.2``, which inflates
  GARCH's RMSE by roughly that amount regardless of how good the model is. Most
  of the "+30.8%" is a unit error, not skill.

The honest bar for a daily realized-volatility forecast is **HAR-RV**
(Corsi 2009): an OLS regression of tomorrow's log-RV on daily, weekly and
monthly averages of log-RV. It is three lines of code, has no hyperparameters,
and is genuinely hard to beat. If LightGBM plus FinBERT cannot beat HAR, the
project's headline claim does not survive.
"""


import numpy as np
import pandas as pd

HAR_LAGS = {"d": 1, "w": 5, "m": 22}


# --------------------------------------------------------------------------
# naive persistence
# --------------------------------------------------------------------------

def naive_forecast(log_rv: pd.Series) -> pd.Series:
    """Tomorrow's log-vol = today's log-"""
    return log_rv.copy()


# --------------------------------------------------------------------------
# HAR-RV
# --------------------------------------------------------------------------

def har_design(log_rv: pd.Series) -> pd.DataFrame:
    """HAR regressors at date t (all known at t's close)."""
    return pd.DataFrame({
        "har_d": log_rv,
        "har_w": log_rv.rolling(HAR_LAGS["w"]).mean(),
        "har_m": log_rv.rolling(HAR_LAGS["m"]).mean(),
    }, index=log_rv.index)


class HARModel:
    """OLS HAR-RV, fit by least squares with an intercept."""

    def __init__(self) -> None:
        self.coef_: np.ndarray | None = None
        self.columns_ = ["har_d", "har_w", "har_m"]

    def fit(self, X: pd.DataFrame, y: pd.Series) -> "HARModel":
        design = X[self.columns_]
        mask = design.notna().all(axis=1) & y.notna()
        A = np.column_stack([np.ones(mask.sum()), design.loc[mask].to_numpy()])
        self.coef_, *_ = np.linalg.lstsq(A, y.loc[mask].to_numpy(), rcond=None)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        if self.coef_ is None:
            raise RuntimeError("HARModel.fit must be called before predict")
        design = X[self.columns_].to_numpy()
        A = np.column_stack([np.ones(len(design)), design])
        # Rows with a NaN regressor fall back to the intercept-only prediction.
        out = A @ self.coef_
        bad = ~np.isfinite(A).all(axis=1)
        out[bad] = self.coef_[0]
        return out


# --------------------------------------------------------------------------
# GARCH(1,1)
# --------------------------------------------------------------------------

def garch_forecast(log_returns: pd.Series, refit_every: int = 5,
                   min_history: int = 250) -> pd.Series:
    """Rolling one-step-ahead GARCH(1,1) forecast of *total* daily log-

    Indexed so that the value at date t is the forecast for t+1 -- the same
    convention as every other forecast in this project, so no caller-side
    ``shift`` is required (v1 pushed that onto the caller, which is exactly the
    kind of thing that silently goes wrong).

    Returns log-vol in the same close-to-close units as ``target_log_rv_h1``,
    so the comparison is now apples to apples.
    """
    from arch import arch_model

    rets = (log_returns.dropna() * 100.0)
    out = pd.Series(index=rets.index, dtype=float)

    last_params = None
    for i in range(min_history, len(rets)):
        history = rets.iloc[:i]
        try:
            model = arch_model(history, vol="GARCH", p=1, q=1, mean="Zero",
                               rescale=False)
            if last_params is None or (i - min_history) % refit_every == 0:
                res = model.fit(disp="off")
                last_params = res.params
            else:
                res = model.fix(last_params)
            var = res.forecast(horizon=1, reindex=False).variance.values[-1, 0]
            # forecast made from data strictly before rets.index[i] is a
            # forecast *for* rets.index[i]; store it at i-1 so the index
            # convention is "value at t forecasts t+1".
            out.iloc[i - 1] = np.log(np.sqrt(var) / 100.0)
        except Exception:
            continue

    return out.reindex(log_returns.index)


def garch_bias_correction(garch_log: pd.Series, target_log: pd.Series,
                          train_mask: np.ndarray) -> float:
    """Mean log offset between GARCH and the target, estimated on training data.

    Even with matched units a one-observation realized-vol target sits below a
    conditional-vol forecast in expectation (Jensen, plus estimator noise). We
    remove that offset using training data only, so GARCH is scored on its
    *shape*, which is what a baseline comparison is supposed to test.
    """
    a = garch_log[train_mask]
    b = target_log[train_mask]
    mask = a.notna() & b.notna()
    if mask.sum() < 30:
        return 0.0
    return float((b[mask] - a[mask]).mean())

In [ ]:
# ===== src/model.py =====
"""LightGBM volatility model.

One real bug fixed from v1. v1 tuned hyperparameters with early stopping on a
validation tail, took ``n_estimators`` straight out of the trial's *suggested*
value, and then refit on the full training window **with no early stopping**:

    best_params = tune_hyperparameters(X_tr, y_tr, X_val, y_val, ...)
    model = train_lgbm(train_df[feature_cols], train_df[target], params=best_params)

So the number of trees that survived early stopping during tuning was thrown
away, and the refit ran the full suggested ``n_estimators`` -- up to 500 -- on
data it had never been validated against. The tuned configuration and the
deployed model were not the same model. Here the surviving iteration count is
carried through and rescaled for the larger refit sample.
"""


import numpy as np
import pandas as pd

RANDOM_SEED = 42

BASE_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "seed": RANDOM_SEED,
    "n_jobs": -1,
}


def _search_space(trial) -> dict:
    return {
        "num_leaves": trial.suggest_int("num_leaves", 7, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": 2000,  # capped by early stopping, not by the search
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }


def tune(X_tr, y_tr, X_val, y_val, n_trials: int = 25) -> dict:
    """Optuna search; returns params including the early-stopped tree count."""
    import lightgbm as lgb
    import optuna

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    best_iters: dict[int, int] = {}

    def objective(trial):
        params = {**BASE_PARAMS, **_search_space(trial)}
        model = lgb.LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        best_iters[trial.number] = int(model.best_iteration_ or params["n_estimators"])
        pred = model.predict(X_val, num_iteration=model.best_iteration_)
        return float(np.sqrt(np.mean((pred - np.asarray(y_val)) ** 2)))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    params = {**BASE_PARAMS, **_search_space_from_dict(study.best_params)}
    params["n_estimators"] = best_iters.get(study.best_trial.number, 200)
    return params


def _search_space_from_dict(d: dict) -> dict:
    out = dict(d)
    out["subsample_freq"] = 1
    return out


def fit(X, y, params: dict, val_fraction: float = 0.0):
    """Fit LightGBM, rescaling the tree count for the larger refit sample.

    ``params['n_estimators']`` arrives as the count that survived early stopping
    on ``val_fraction``-smaller data; scaling it up keeps the effective amount
    of boosting roughly constant on the full window.
    """
    import lightgbm as lgb

    p = dict(params)
    if val_fraction > 0:
        p["n_estimators"] = max(20, int(p.get("n_estimators", 200) / (1 - val_fraction)))
    model = lgb.LGBMRegressor(**p)
    model.fit(X, y)
    return model


def fit_quantiles(X, y, params: dict, quantiles=(0.1, 0.5, 0.9),
                  val_fraction: float = 0.0) -> dict:
    """One model per quantile for a cheap predictive band."""
    models = {}
    for q in quantiles:
        p = dict(params)
        p["objective"] = "quantile"
        p["alpha"] = q
        p.pop("metric", None)
        models[q] = fit(X, y, p, val_fraction)
    return models

## 7. Walk-forward validation

Expanding window, never a shuffle. Two additions over v1:

- **Diebold-Mariano** with Newey-West standard errors, so "beats HAR" is a claim
  with a p-value rather than a bare percentage over noisy folds.
- **Purging** of overlapping multi-day targets at the train/test boundary —
  without it the `h=5` model trains on labels built from the data it is about to
  be scored on.

In [ ]:
# ===== src/validation.py =====
"""Walk-forward validation, with a significance test instead of a bare percentage.

v1's headline was "+15.0% RMSE vs naive". Across 24 folds of a noisy daily
series that number carries no error bar, and a percentage improvement over a
deliberately weak baseline is the easiest kind of result to produce by
accident. This module adds:

  * **HAR-RV** as the baseline that actually matters (see src/py);
  * a **Diebold-Mariano test** on the squared-error differential, with
    Newey-West standard errors, so "the model beats HAR" becomes a claim with a
    p-value attached;
  * per-fold smearing factors estimated on training residuals, so the level
    forecasts handed to the backtest are unbiased in the mean rather than the
    median (this is the correction v1 omitted -- see volatility.smearing_factor).
"""


import re
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta



@dataclass
class Fold:
    fold: int
    train_start: pd.Timestamp
    train_end: pd.Timestamp
    test_start: pd.Timestamp
    test_end: pd.Timestamp
    n_train: int
    n_test: int
    smearing: float
    predictions: pd.DataFrame = field(repr=False)


def generate_folds(dates: pd.DatetimeIndex, train_years: int = 2,
                   test_months: int = 2, step_months: int = 2,
                   purge: int = 0):
    """Expanding-window folds. Never a shuffle split.

    ``purge`` drops the last ``purge`` rows of each training window. With a
    multi-day target this is mandatory: the target at the final training date
    is an average over days that fall inside the test window, so without
    purging the model is trained on labels built from the data it is about to
    be scored on. For ``target_*_h{h}`` the correct value is ``h - 1``.
    """
    dates = pd.DatetimeIndex(dates)
    start = dates.min()
    train_end = start + relativedelta(years=train_years)
    while True:
        test_end = train_end + relativedelta(months=test_months)
        train_mask = (dates >= start) & (dates < train_end)
        test_mask = (dates >= train_end) & (dates < test_end)
        if purge > 0 and train_mask.sum() > purge:
            keep = np.flatnonzero(train_mask)[:-purge]
            purged = np.zeros_like(train_mask)
            purged[keep] = True
            train_mask = purged
        if test_mask.sum() == 0:
            break
        if train_mask.sum() > 0:
            yield train_mask, test_mask
        train_end = train_end + relativedelta(months=step_months)
        if train_end >= dates.max():
            break


def run_walk_forward(table: pd.DataFrame, feature_cols: list[str],
                     target_col: str = "target_log_rv_h1",
                     garch_log: pd.Series | None = None,
                     train_years: int = 2, test_months: int = 2,
                     step_months: int = 2, n_trials: int = 25,
                     retune_every: int = 4, val_fraction: float = 0.15,
                     purge: int | None = None,
                     verbose: bool = True) -> list[Fold]:
    """Expanding walk-forward over LightGBM, HAR, naive and (optionally) GARCH.

    ``retune_every`` re-runs Optuna only every k folds and reuses the params in
    between. v1 ran a fresh 20-trial study on all 24 folds, which is both slow
    and a source of fold-to-fold noise that has nothing to do with the signal.

    ``purge`` defaults to ``h - 1`` inferred from ``target_col``, which is what
    a multi-day overlapping target requires. See generate_folds.
    """
    if purge is None:
        m = re.search(r"_h(\d+)$", target_col)
        purge = (int(m.group(1)) - 1) if m else 0
    df = table.dropna(subset=[target_col]).copy()
    dates = df.index
    log_rv = df["log_rv"]
    har_X = har_design(log_rv)

    folds: list[Fold] = []
    params: dict | None = None

    for i, (train_mask, test_mask) in enumerate(
            generate_folds(dates, train_years, test_months, step_months, purge)):
        train_df, test_df = df.loc[train_mask], df.loc[test_mask]
        if len(train_df) < 120 or len(test_df) == 0:
            continue

        X_tr_all, y_tr_all = train_df[feature_cols], train_df[target_col]
        cut = int(len(train_df) * (1 - val_fraction))
        X_tr, y_tr = X_tr_all.iloc[:cut], y_tr_all.iloc[:cut]
        X_val, y_val = X_tr_all.iloc[cut:], y_tr_all.iloc[cut:]

        if params is None or i % retune_every == 0:
            params = tune(X_tr, y_tr, X_val, y_val, n_trials=n_trials)

        lgbm = fit(X_tr_all, y_tr_all, params, val_fraction=val_fraction)

        # Smearing estimated on the held-out validation tail of the *training*
        # window -- in-sample residuals would understate it badly.
        val_resid = np.asarray(y_val) - lgbm.predict(X_val)
        smear = smearing_factor(val_resid)

        har = HARModel().fit(har_X.loc[train_mask], y_tr_all)

        preds = pd.DataFrame({
            "date": test_df.index,
            "y_true": test_df[target_col].to_numpy(),
            "model": lgbm.predict(test_df[feature_cols]),
            "naive": naive_forecast(log_rv.loc[test_mask]).to_numpy(),
            "har": har.predict(har_X.loc[test_mask]),
        })

        if garch_log is not None:
            offset = garch_bias_correction(
                garch_log.reindex(df.index), df[target_col], train_mask)
            preds["garch"] = garch_log.reindex(test_df.index).to_numpy() + offset
        else:
            preds["garch"] = np.nan

        folds.append(Fold(
            fold=len(folds),
            train_start=train_df.index.min(), train_end=train_df.index.max(),
            test_start=test_df.index.min(), test_end=test_df.index.max(),
            n_train=len(train_df), n_test=len(test_df),
            smearing=smear, predictions=preds,
        ))
        if verbose:
            print(f"  fold {folds[-1].fold:2d}  "
                  f"test {preds['date'].min().date()} -> {preds['date'].max().date()}  "
                  f"n={len(test_df):3d}  smearing={smear:.3f}")

    return folds


# --------------------------------------------------------------------------
# metrics
# --------------------------------------------------------------------------

def stack_predictions(folds: list[Fold]) -> pd.DataFrame:
    out = pd.concat([f.predictions for f in folds], ignore_index=True)
    smear = pd.concat([
        pd.Series(f.smearing, index=range(len(f.predictions))) for f in folds
    ], ignore_index=True)
    out["smearing"] = smear.to_numpy()
    return out.set_index("date").sort_index()


def error_table(preds: pd.DataFrame,
                cols=("model", "har", "naive", "garch")) -> pd.DataFrame:
    rows = []
    for c in cols:
        if c not in preds or preds[c].isna().all():
            continue
        mask = preds[c].notna() & preds["y_true"].notna()
        err = preds.loc[mask, c] - preds.loc[mask, "y_true"]
        rows.append({
            "forecast": c,
            "rmse": float(np.sqrt((err**2).mean())),
            "mae": float(err.abs().mean()),
            "n": int(mask.sum()),
        })
    return pd.DataFrame(rows).set_index("forecast")


def diebold_mariano(preds: pd.DataFrame, a: str = "model", b: str = "har",
                    lag: int | None = None) -> dict:
    """DM test on the squared-error differential ``e_a^2 - e_b^2``.

    Negative statistic => forecast ``a`` has the lower loss. Newey-West
    standard errors, since daily volatility errors are serially correlated and
    a naive t-stat would overstate significance.
    """
    mask = preds[a].notna() & preds[b].notna() & preds["y_true"].notna()
    ea = (preds.loc[mask, a] - preds.loc[mask, "y_true"]).to_numpy()
    eb = (preds.loc[mask, b] - preds.loc[mask, "y_true"]).to_numpy()
    d = ea**2 - eb**2
    T = len(d)
    if T < 30:
        return {"statistic": np.nan, "p_value": np.nan, "n": T}

    if lag is None:
        lag = int(np.floor(4 * (T / 100.0) ** (2.0 / 9.0)))
    dbar = d.mean()
    dc = d - dbar
    gamma0 = float(dc @ dc / T)
    var = gamma0
    for k in range(1, lag + 1):
        gk = float(dc[k:] @ dc[:-k] / T)
        var += 2.0 * (1.0 - k / (lag + 1.0)) * gk
    var = max(var, 1e-18)

    stat = dbar / np.sqrt(var / T)
    # two-sided normal p-value
    from math import erfc, sqrt
    p = erfc(abs(stat) / sqrt(2.0))
    return {"statistic": float(stat), "p_value": float(p), "n": T,
            "mean_loss_diff": float(dbar), "lag": lag}

In [ ]:
RUN_GARCH = False   # rolling GARCH over 10y is slow; flip on for the full table
garch_log = garch_forecast(table["log_return"], refit_every=25) if RUN_GARCH else None

folds = run_walk_forward(table, cols, target_col="target_log_rv_h1",
                          garch_log=garch_log, train_years=TRAIN_YEARS,
                          test_months=TEST_MONTHS, step_months=TEST_MONTHS,
                          n_trials=N_TRIALS)
preds = stack_predictions(folds)
errs = error_table(preds)
print("\n", errs, "\n")

for name in ["har", "naive"]:
    if name in errs.index:
        b = errs.loc[name, "rmse"]
        print(f"Model vs {name:>5}: {100*(b-errs.loc['model','rmse'])/b:+.1f}% RMSE")

dm = diebold_mariano(preds, "model", "har")
print(f"\nDiebold-Mariano vs HAR-RV: stat={dm['statistic']:+.2f}  p={dm['p_value']:.4f}  n={dm['n']}")
print("->", "model significantly better" if dm["p_value"] < 0.05 and dm["statistic"] < 0
      else "NOT significant -- the model does not beat HAR")

In [ ]:
folds5 = run_walk_forward(table, cols, target_col="target_log_rv_h5",
                           train_years=TRAIN_YEARS, test_months=TEST_MONTHS,
                           step_months=TEST_MONTHS, n_trials=N_TRIALS, verbose=False)
preds5 = stack_predictions(folds5)
print(error_table(preds5))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(preds.index, np.exp(preds["y_true"]) * np.sqrt(252) * 100,
        lw=1.0, alpha=0.75, label="Realized (gap-inclusive)")
ax.plot(preds.index, np.exp(preds["model"]) * preds["smearing"] * np.sqrt(252) * 100,
        lw=1.2, label="LightGBM forecast")
ax.plot(preds.index, table["iv_daily"].reindex(preds.index) * np.sqrt(252) * 100,
        lw=1.0, alpha=0.8, label="India VIX (implied)")
ax.set_title("Walk-forward: forecast vs realized vs implied (annualized %)")
ax.set_ylabel("vol %"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 8. Does sentiment contribute? (ablation, not SHAP share)

v1 reported "sentiment contributes 0.0% of SHAP" and stopped there. But SHAP
share describes *this fitted model*, not the data — when sentiment features are
noisy and partly collinear with price features, greedy tree splitting never
selects them and their SHAP mass is zero by construction.

The decision-useful question is whether out-of-sample error gets worse without
them. That is an ablation with a p-value. It may still come back negative, and
a tested negative is a better result than an untestable SHAP share.

In [ ]:
# ===== src/explain.py =====
"""SHAP attribution, and a fairer test of whether sentiment contributes.

v1 reported "sentiment contributes 0.0% of total feature importance" and left
it there. Two things are wrong with stopping at that number.

First, |SHAP| share is a statement about *this fitted model*, not about the
data. When sentiment features are collinear with price features and arrive
second in the tree-building order, LightGBM will simply never split on them and
their SHAP mass is zero by construction. That is a fact about greedy splitting,
not evidence that news carries no information.

Second, the decision-useful question is not "what share of importance" but
"does out-of-sample error get worse if I remove these features". That is a
one-line experiment and it is the one that settles the argument.
``incremental_value`` runs it.
"""


import numpy as np
import pandas as pd

SENTIMENT_PREFIXES = ("sent_", "news_vol", "article_count", "pct_", "has_news")
VIX_PREFIXES = ("log_iv", "log_vrp", "vix_")


def compute_shap(model, X: pd.DataFrame):
    import shap
    return shap.TreeExplainer(model)(X)


def importance(shap_values, feature_names: list[str]) -> pd.DataFrame:
    mean_abs = np.abs(shap_values.values).mean(axis=0)
    return (pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True))


def _group(name: str) -> str:
    if name.startswith(SENTIMENT_PREFIXES):
        return "sentiment"
    if name.startswith(VIX_PREFIXES):
        return "implied_vol"
    return "price"


def group_shares(importance_df: pd.DataFrame) -> pd.Series:
    g = importance_df.assign(group=importance_df["feature"].map(_group))
    total = g["mean_abs_shap"].sum()
    if total == 0:
        return pd.Series(dtype=float)
    return (100 * g.groupby("group")["mean_abs_shap"].sum() / total
            ).sort_values(ascending=False)


def incremental_value(table: pd.DataFrame, base_cols: list[str],
                      extra_cols: list[str], target_col: str = "target_log_rv_h1",
                      **wf_kwargs) -> dict:
    """Does adding ``extra_cols`` reduce walk-forward RMSE? With a p-value.

    This is the honest replacement for "sentiment contributes 0.0% of SHAP".
    Runs the same walk-forward twice -- once without the extra block, once with
    it -- and Diebold-Mariano tests the difference in squared error. If the
    p-value is not small, the extra features do not help, and that is a real
    finding worth reporting rather than something to hide.
    """

    wf_kwargs.setdefault("verbose", False)
    base = stack_predictions(
        run_walk_forward(table, base_cols, target_col, **wf_kwargs))
    full = stack_predictions(
        run_walk_forward(table, base_cols + extra_cols, target_col,
                                    **wf_kwargs))

    joined = pd.DataFrame({
        "y_true": base["y_true"],
        "base": base["model"],
        "full": full["model"].reindex(base.index),
    }).dropna()

    rmse_base = float(np.sqrt(((joined["base"] - joined["y_true"]) ** 2).mean()))
    rmse_full = float(np.sqrt(((joined["full"] - joined["y_true"]) ** 2).mean()))
    dm = diebold_mariano(joined, "full", "base")

    return {
        "rmse_without": rmse_base,
        "rmse_with": rmse_full,
        "improvement_pct": 100 * (rmse_base - rmse_full) / rmse_base,
        "dm_statistic": dm["statistic"],
        "p_value": dm["p_value"],
        "n": dm["n"],
        "verdict": ("adds real information" if dm["statistic"] < 0 and dm["p_value"] < 0.05
                    else "no significant contribution"),
    }

In [ ]:
final = table.dropna(subset=["target_log_rv_h1"])
cut = int(len(final) * 0.85)
params = tune(final[cols].iloc[:cut], final["target_log_rv_h1"].iloc[:cut],
              final[cols].iloc[cut:], final["target_log_rv_h1"].iloc[cut:],
              n_trials=N_TRIALS)
final_model = fit(final[cols], final["target_log_rv_h1"], params, val_fraction=0.15)

imp = importance(compute_shap(final_model, final[cols].iloc[-500:]), cols)
print(group_shares(imp).round(1).to_string(), "\n")
print(imp.head(15).to_string(index=False))

In [ ]:
sent_cols = [c for c in cols if c.startswith(("sent_", "news_vol", "has_news"))]
if sent_cols:
    base_cols = [c for c in cols if c not in sent_cols]
    print("SENTIMENT ablation:", incremental_value(
        table, base_cols, sent_cols, n_trials=10, train_years=TRAIN_YEARS))
else:
    print("No sentiment features (RUN_NEWS was False).")

vix_cols = [c for c in cols if c.startswith(("log_iv", "log_vrp", "vix_"))]
if vix_cols:
    base_cols = [c for c in cols if c not in vix_cols]
    print("\nINDIA VIX ablation:", incremental_value(
        table, base_cols, vix_cols, n_trials=10, train_years=TRAIN_YEARS))

## 9. Directional signals — the prerequisite for shorting

A volatility model cannot tell you which way to bet: it forecasts `|r|`, and the
sign is exactly what it discards. So "add shorting" means "add a return
forecast", which is a harder problem — realized volatility is strongly
autocorrelated and forecastable, daily index returns are close to a martingale.

Three cheap causal signals: time-series momentum (the only one with decades of
out-of-sample evidence), the variance risk premium as a risk-appetite proxy, and
signed news polarity.

**Read the IC table before you look at any equity curve.** If the blend's t-stat
is under 2, the long/short backtest below is a draw from noise no matter how
good its Sharpe looks.

In [ ]:
# ===== src/signals.py =====
"""Directional signals -- what you need before you can short anything.

A volatility model cannot tell you which way to bet. It forecasts |r|, not r,
and the sign is exactly the information it throws away. So "add shorting" is
really "add a return forecast", and that is a different and much harder
problem: realized volatility is strongly autocorrelated and therefore
forecastable, while daily index returns are close to a martingale.

Set expectations accordingly. Everything in this module has a documented
out-of-sample information coefficient in the 0.02-0.05 range at daily horizon.
That is small but not zero, and it is the honest ceiling. Anything claiming
much more on an index this liquid is overfit.

Three signals, all cheap, all causal:

* **Time-series momentum.** Distance from a moving average, normalised by the
  volatility accumulated over that lookback. The most robust directional
  anomaly in the literature and the only one here you would trade on its own.
* **Variance risk premium.** A wide VRP predicts higher subsequent equity
  returns (Bollerslev-Tauchen-Zhou 2009) -- it is a risk-appetite proxy. Free,
  since India VIX is already loaded for the vol model.
* **News sentiment.** Signed mean polarity: useless for forecasting volatility
  (section 3 of CRITICAL_ANALYSIS.md) because good and bad news both raise it,
  but direction is the one question the sign is actually about. This is where
  the FinBERT work belongs.

Every statistic here is computed on a trailing window. Z-scoring against
full-sample mean and standard deviation is lookahead bias and would inflate
every number in this file.
"""


import numpy as np
import pandas as pd

TRADING_DAYS = 252


def _rolling_z(series: pd.Series, window: int = TRADING_DAYS,
               min_periods: int = 60, clip: float = 3.0) -> pd.Series:
    """Trailing z-score. Causal by construction."""
    mu = series.rolling(window, min_periods=min_periods).mean()
    sd = series.rolling(window, min_periods=min_periods).std()
    return ((series - mu) / sd.replace(0, np.nan)).clip(-clip, clip)


def trend_score(close: pd.Series, daily_vol: pd.Series,
                spans: tuple[int, ...] = (20, 60, 200)) -> pd.Series:
    """Multi-horizon time-series momentum, volatility-normalised, in [-1, 1].

    For each lookback ``n``: ``log(P_t / MA_n) / (sigma * sqrt(n))``. The
    ``sqrt(n)`` matters -- over ``n`` days the price accumulates about
    ``sigma * sqrt(n)`` of noise, so without it the long lookbacks dominate
    purely by scale rather than by information. ``tanh`` then caps each
    horizon's vote so one runaway trend cannot swamp the blend.
    """
    votes = []
    for n in spans:
        ma = close.rolling(n, min_periods=n // 2).mean()
        z = np.log(close / ma) / (daily_vol * np.sqrt(n)).replace(0, np.nan)
        votes.append(np.tanh(z))
    return pd.concat(votes, axis=1).mean(axis=1)


def vrp_score(log_vrp: pd.Series, window: int = TRADING_DAYS) -> pd.Series:
    """Risk-appetite proxy in [-1, 1]. Wide premium -> higher expected returns."""
    return np.tanh(_rolling_z(log_vrp, window) / 2.0)


def sentiment_score(sent_mean: pd.Series, window: int = TRADING_DAYS) -> pd.Series:
    """Signed news polarity, trailing-standardised, in [-1, 1].

    NaN on days with no news -- filled with 0 (neutral) here rather than left
    NaN, because unlike the tree model this is an arithmetic blend and a NaN
    would poison the whole score.
    """
    return np.tanh(_rolling_z(sent_mean, window) / 2.0).fillna(0.0)


def build_direction(table: pd.DataFrame,
                    weights: dict[str, float] | None = None,
                    allow_short: bool = True,
                    short_floor: float = -1.0,
                    target_exposure: float = 0.7) -> pd.DataFrame:
    """Blend the available signals into one directional score per day.

    Returns a frame with the individual components plus ``direction``, the
    weighted blend clipped to ``[short_floor, 1]``. With ``allow_short=False``
    the floor is 0 and the strategy degenerates to the long-only overlay.

    Weights are *not* fitted. Fitting three signal weights on the same sample
    you then report is how a 0.03 information coefficient becomes an imaginary
    0.15; the defaults lean on momentum because that is the one with decades of
    out-of-sample evidence behind it.

    ``target_exposure`` rescales the blend so its trailing mean absolute value
    is about that much. Three ``tanh``-squashed components averaged together
    sit near zero most of the time, which would leave the book ~3% invested and
    ~97% in cash -- a strategy whose returns are almost entirely the risk-free
    rate, wearing a flattering Sharpe. The rescaling is done on a trailing
    window so it stays causal.
    """
    weights = weights or {"trend": 0.6, "vrp": 0.25, "sentiment": 0.15}

    out = pd.DataFrame(index=table.index)
    out["trend"] = trend_score(table["close"], table["rv"])

    if "log_vrp" in table and table["log_vrp"].notna().any():
        out["vrp"] = vrp_score(table["log_vrp"])
    else:
        out["vrp"] = np.nan

    if "sent_mean" in table and table["sent_mean"].notna().any():
        out["sentiment"] = sentiment_score(table["sent_mean"])
    else:
        out["sentiment"] = np.nan

    # Renormalise over whichever components actually exist, so a missing VIX or
    # missing news feed rescales the blend instead of silently shrinking it.
    w = pd.Series({k: weights.get(k, 0.0) for k in ["trend", "vrp", "sentiment"]})
    available = out[["trend", "vrp", "sentiment"]].notna()
    wsum = available.mul(w, axis=1).sum(axis=1).replace(0, np.nan)
    blend = out[["trend", "vrp", "sentiment"]].fillna(0.0).mul(w, axis=1).sum(axis=1) / wsum

    scale = blend.abs().rolling(TRADING_DAYS, min_periods=60).mean()
    blend = blend * (target_exposure / scale.replace(0, np.nan))

    lo = short_floor if allow_short else 0.0
    out["direction"] = blend.clip(lo, 1.0).fillna(0.0)
    return out


# --------------------------------------------------------------------------
# does the signal actually predict anything?
# --------------------------------------------------------------------------

def information_coefficient(signal: pd.Series, next_return: pd.Series,
                            lag: int | None = None) -> dict:
    """Rank correlation between signal and next-day return, with a t-stat.

    Run this *before* backtesting. A directional signal with an insignificant
    IC will still produce an equity curve -- it just will not produce one that
    means anything, and the backtest's Sharpe will be a draw from noise. The
    t-stat uses Newey-West standard errors because both the signal and the
    returns are serially correlated.
    """
    df = pd.DataFrame({"s": signal, "r": next_return}).dropna()
    if len(df) < 60:
        return {"ic": np.nan, "t_stat": np.nan, "p_value": np.nan, "n": len(df)}

    ic = float(df["s"].corr(df["r"], method="spearman"))

    # t-stat on the per-period contribution, Newey-West corrected.
    x = df["s"].rank(pct=True) - 0.5
    y = df["r"].rank(pct=True) - 0.5
    prod = (x * y).to_numpy()
    T = len(prod)
    if lag is None:
        lag = int(np.floor(4 * (T / 100.0) ** (2.0 / 9.0)))
    dbar = prod.mean()
    dc = prod - dbar
    var = float(dc @ dc / T)
    for k in range(1, lag + 1):
        var += 2.0 * (1.0 - k / (lag + 1.0)) * float(dc[k:] @ dc[:-k] / T)
    var = max(var, 1e-18)
    t = dbar / np.sqrt(var / T)

    from math import erfc, sqrt
    return {
        "ic": ic,
        "t_stat": float(t),
        "p_value": float(erfc(abs(t) / sqrt(2.0))),
        "n": T,
        "verdict": ("has predictive power" if abs(t) > 1.96
                    else "NOT distinguishable from noise"),
    }


def signal_report(direction_df: pd.DataFrame, next_return: pd.Series) -> pd.DataFrame:
    """IC table for every component and the blend. Read it before trading."""
    rows = []
    for col in direction_df.columns:
        stats = information_coefficient(direction_df[col], next_return)
        rows.append({"signal": col, **stats})
    return pd.DataFrame(rows).set_index("signal")

In [ ]:
next_ret = table["log_return"].shift(-1)
direction = build_direction(table, allow_short=True)

print("Information coefficient vs next-day return:\n")
print(signal_report(direction.reindex(preds.index), next_ret.reindex(preds.index)).round(4))

## 10. Backtests

Three strategies, and they are not equal.

**A — volatility overlay (no direction).** v1's idea, repaired. It will not make
money and is not supposed to: Sharpe is invariant to leverage, so the only thing
that moves it is `Cov(1/sigma_hat, r_next)` — and the model forecasts volatility,
not returns. Claim the drawdown, not the return.

**B — long/short.** Direction from section 9, size from the volatility model.
This is the only arrangement where the volatility forecast contributes to return,
and it does so indirectly: it makes each unit of directional conviction carry
constant risk. Shorting is via futures, so the carry arithmetic differs — you
give up the dividend yield and fight the index's drift.

**C — variance risk premium carry.** Where a volatility forecast genuinely pays.
India VIX prices implied vol, the model forecasts realized vol, and the spread is
tradeable. This converts the model from a position-sizing input into a *pricing*
input, where every RMSE improvement maps onto P&L.

In [ ]:
# ===== src/backtest.py =====
"""Backtests.

Two strategies, for two different reasons.

``vol_target_backtest`` is v1's idea, repaired. It will not make money and it
is not supposed to -- a long-only position sized by a volatility forecast has
no expected-return edge, because Sharpe is invariant to leverage. The most it
can do is reshape risk: shallower drawdowns for a similar return. v1 reported
it as a profit strategy and lost to buy-and-hold on every axis; fixed, it
should land at a modestly better Sharpe and a much better drawdown, with the
honest caveat attached.

``vrp_backtest`` is where a volatility forecast actually pays. India VIX prices
*implied* volatility; the model forecasts *realized* volatility; the spread
between them is the variance risk premium, which on the Nifty is large and
persistently positive. Selling variance harvests it, and the model's job is to
tell you when the premium is unusually rich (size up) or thin/negative (stand
aside, or buy). This converts the volatility forecast from a position-sizing
input, where its skill is nearly worthless, into a pricing input, where its
skill is the whole trade.

The short-variance leg has severe negative skew. Everything here is capped and
limited, and the caveats are in CRITICAL_ANALYSIS.md section 5 -- read them
before believing any Sharpe this file prints.
"""


import numpy as np
import pandas as pd

TRADING_DAYS = 252


# --------------------------------------------------------------------------
# shared metrics
# --------------------------------------------------------------------------

def sharpe(returns: pd.Series, rf_annual: float = 0.0) -> float:
    r = pd.Series(returns).dropna()
    if len(r) < 2 or r.std() == 0:
        return 0.0
    excess = r - rf_annual / TRADING_DAYS
    return float(np.sqrt(TRADING_DAYS) * excess.mean() / r.std())


def max_drawdown(equity: pd.Series) -> float:
    eq = pd.Series(equity).dropna()
    if eq.empty:
        return 0.0
    return float((eq / eq.cummax() - 1.0).min())


def summarize(returns: pd.Series, rf_annual: float = 0.0,
              label: str = "") -> dict:
    r = pd.Series(returns).dropna()
    equity = (1 + r).cumprod()
    years = len(r) / TRADING_DAYS
    cagr = float(equity.iloc[-1] ** (1 / years) - 1) if years > 0 and len(equity) else 0.0

    # Sortino must use the same excess return as Sharpe. Comparing an
    # excess-return Sharpe against a raw-return Sortino produces contradictory
    # signs on the same series and makes a cash-heavy book look good.
    excess = r - rf_annual / TRADING_DAYS
    downside = excess[excess < 0].std()
    return {
        "label": label,
        "cagr": cagr,
        "ann_vol": float(r.std() * np.sqrt(TRADING_DAYS)),
        "sharpe": sharpe(r, rf_annual),
        "sortino": float(np.sqrt(TRADING_DAYS) * excess.mean() / downside) if downside else 0.0,
        "max_drawdown": max_drawdown(equity),
        "total_return": float(equity.iloc[-1] - 1) if len(equity) else 0.0,
        "skew": float(r.skew()),
        "worst_day": float(r.min()),
    }


def _apply_deadband(raw: pd.Series, deadband: float) -> pd.Series:
    """No-trade band with partial adjustment to the band edge.

    v1 rebalanced to a fresh target every single day off a noisy daily forecast
    and paid 7.5 bps on every wiggle. Backing out v1's own reported numbers,
    that cost drag was ~2-3.5%/yr -- larger than any plausible vol-timing
    benefit, and the single biggest reason the strategy lost to buy-and-hold.

    Snapping to the full target whenever the band is breached barely helps,
    because a noisy target breaches it most days. Trading only to the *edge* of
    the band is the classic no-trade-region result (Leland; Davis-Norman): the
    book tracks the target with a lag, turnover collapses, and the tracking
    error costs far less than the spread it saves.
    """
    if deadband <= 0:
        return raw.ffill().fillna(0.0)

    out = np.empty(len(raw))
    current = float(raw.iloc[0]) if len(raw) else 0.0
    for i, target in enumerate(raw.to_numpy()):
        if np.isfinite(target):
            if target > current + deadband:
                current = float(target) - deadband
            elif target < current - deadband:
                current = float(target) + deadband
        out[i] = current
    return pd.Series(out, index=raw.index)


# --------------------------------------------------------------------------
# strategy 1: volatility-targeted index exposure (the repaired v1 idea)
# --------------------------------------------------------------------------

def vol_target_backtest(pred_vol: pd.Series, next_return: pd.Series,
                        target_ann_vol: float = 0.12,
                        max_leverage: float = 1.5,
                        cost_bps: float = 7.5,
                        rf_annual: float = 0.065,
                        deadband: float = 0.10) -> pd.DataFrame:
    """Long-only index exposure scaled inversely to predicted volatility.

    ``pred_vol`` must be a **gap-inclusive daily** vol forecast with the
    smearing correction already applied -- the same units as the close-to-close
    returns in ``next_return``. Getting this wrong is what produced v1's
    persistent 1.3x leverage.

    Unlike v1 this charges financing on borrowed exposure and credits cash on
    un-invested capital. v1 handed the strategy free leverage, which flatters
    it, and it still lost.
    """
    idx = pred_vol.index.intersection(next_return.index)
    pv = pred_vol.loc[idx].astype(float)
    ret = next_return.loc[idx].astype(float)

    target_daily = target_ann_vol / np.sqrt(TRADING_DAYS)
    raw = (target_daily / pv.replace(0, np.nan)).clip(0.0, max_leverage)
    pos = _apply_deadband(raw.ffill().fillna(0.0), deadband)

    traded = pos.diff().abs().fillna(pos.abs())
    cost = traded * (cost_bps / 1e4)
    # borrow above 1x, earn cash below 1x
    financing = (pos - 1.0) * (rf_annual / TRADING_DAYS)

    strat = pos * ret - cost - financing

    out = pd.DataFrame({
        "predicted_vol": pv,
        "position": pos,
        "target_position": raw,
        "next_return": ret,
        "cost": cost,
        "financing": financing,
        "strategy_return": strat,
        "buy_hold_return": ret,
    })
    out["strategy_equity"] = (1 + out["strategy_return"]).cumprod()
    out["buy_hold_equity"] = (1 + out["buy_hold_return"]).cumprod()
    return out


def futures_carry(position: pd.Series, rf_annual: float = 0.065,
                  div_yield: float = 0.012) -> pd.Series:
    """Daily carry for a fully-collateralised Nifty futures position.

    You cannot short the cash index in India; shorting means futures. That
    changes the financing arithmetic and it is worth getting right rather than
    bolting a minus sign onto the long-only version.

    A futures price is ``F = S * exp((r - q) * T)``, so holding the future to
    expiry earns the *price* return minus ``(r - q)``. Meanwhile the cash that
    is not posted as margin earns ``r``. Netting the two:

        carry = position * q / 252  +  (1 - position) * r / 252

    Read off the cases: at ``position = 1`` you earn the dividend yield on top
    of the price return, which correctly reconstructs the total return of the
    index (``^NSEI`` is a price index and excludes dividends). At
    ``position = 0`` you earn the risk-free rate on all of it. At
    ``position = -1`` you pay the dividend yield away on the short and collect
    ``r`` on both your own capital and the short proceeds.

    That last line is the one that matters for shorting: the carry is a
    *headwind*, because you are giving up the dividend yield and fighting the
    index's positive drift. A short has to overcome roughly ``q + drift``
    before it breaks even, which is why the long/short version below usually
    looks worse than long-only unless the directional signal is genuinely good.
    """
    return position * (div_yield / TRADING_DAYS) + (1 - position) * (rf_annual / TRADING_DAYS)


def long_short_backtest(direction: pd.Series, pred_vol: pd.Series,
                        next_return: pd.Series,
                        target_ann_vol: float = 0.12,
                        max_long: float = 1.5, max_short: float = -1.0,
                        cost_bps: float = 5.0,
                        rf_annual: float = 0.065, div_yield: float = 0.012,
                        deadband: float = 0.10,
                        stop_drawdown: float | None = -0.25) -> pd.DataFrame:
    """Long/short index exposure: direction from signals, size from the vol model.

    ``position = direction * (target_vol / predicted_vol)``, so the two models
    do separate jobs -- the directional signal picks the side, the volatility
    forecast decides how much risk that side is worth. This is the only
    arrangement in which a volatility forecast contributes to *return* rather
    than only to risk, and it does so indirectly: it makes each unit of
    directional conviction carry constant risk.

    ``cost_bps`` defaults to 5 rather than the 7.5 used for the cash overlay:
    Nifty futures are cheaper to trade (STT on the sell side is 2 bps, spread
    and brokerage roughly another 1-3).

    ``stop_drawdown`` flattens the book if equity falls that far below its high
    water mark, re-entering when the signal next flips. A short book without a
    stop is how accounts get closed; losses on a short are unbounded and margin
    calls arrive at the worst possible moment.
    """
    idx = (direction.index.intersection(pred_vol.index)
           .intersection(next_return.index))
    d = direction.loc[idx].astype(float)
    pv = pred_vol.loc[idx].astype(float)
    ret = next_return.loc[idx].astype(float)

    target_daily = target_ann_vol / np.sqrt(TRADING_DAYS)
    scalar = (target_daily / pv.replace(0, np.nan)).clip(0.0, abs(max_long))
    raw = (d * scalar).clip(max_short, max_long)
    pos = _apply_deadband(raw.ffill().fillna(0.0), deadband)

    if stop_drawdown is not None:
        pos = _apply_drawdown_stop(pos, ret, stop_drawdown, rf_annual, div_yield,
                                   cost_bps)

    traded = pos.diff().abs().fillna(pos.abs())
    cost = traded * (cost_bps / 1e4)
    carry = futures_carry(pos, rf_annual, div_yield)

    strat = pos * ret + carry - cost

    out = pd.DataFrame({
        "direction": d,
        "vol_scalar": scalar,
        "position": pos,
        "next_return": ret,
        "carry": carry,
        "cost": cost,
        "strategy_return": strat,
        "buy_hold_return": ret + div_yield / TRADING_DAYS,
    })
    out["strategy_equity"] = (1 + out["strategy_return"]).cumprod()
    out["buy_hold_equity"] = (1 + out["buy_hold_return"]).cumprod()
    out["gross_exposure"] = pos.abs()
    out["is_short"] = (pos < 0).astype(int)
    return out


def _apply_drawdown_stop(pos: pd.Series, ret: pd.Series, limit: float,
                         rf_annual: float, div_yield: float,
                         cost_bps: float) -> pd.Series:
    """Flatten the book while equity is more than ``limit`` below its peak.

    Walked forward one day at a time because the stop depends on the equity
    curve the stop itself produces -- deriving it from the unstopped curve
    would be a lookahead.

    Re-entry is deliberately conservative: once halted, the book stays flat
    until the target position changes sign relative to what was held when the
    stop fired. Re-entering on the same side that just lost 25% is how a stop
    becomes a formality.
    """
    out = np.zeros(len(pos))
    p = pos.to_numpy()
    r = np.nan_to_num(ret.to_numpy())

    equity, peak = 1.0, 1.0
    halted = False
    halted_side = 0.0
    prev = 0.0

    for i in range(len(p)):
        want = p[i]
        if halted:
            # only wake up when the signal has flipped to the other side
            if halted_side != 0.0 and np.sign(want) == -np.sign(halted_side):
                halted = False
            else:
                want = 0.0

        traded = abs(want - prev)
        carry = (want * (div_yield / TRADING_DAYS)
                 + (1 - want) * (rf_annual / TRADING_DAYS))
        day = want * r[i] + carry - traded * (cost_bps / 1e4)

        equity *= (1 + day)
        peak = max(peak, equity)
        out[i] = want
        prev = want

        if not halted and equity / peak - 1.0 <= limit:
            halted = True
            halted_side = np.sign(want) if want != 0 else np.sign(p[i])

    return pd.Series(out, index=pos.index)


def vol_matched(returns: pd.Series, benchmark: pd.Series) -> pd.Series:
    """Rescale ``returns`` to the benchmark's realized 

    The only fair way to compare total return between a levered and an
    unlevered book. v1 compared a ~1.3x-levered strategy's return against 1x
    buy-and-hold and presented the shortfall as a strategy result.
    """
    a, b = returns.dropna(), benchmark.dropna()
    if a.std() == 0:
        return returns
    return returns * (b.std() / a.std())


# --------------------------------------------------------------------------
# strategy 2: variance risk premium carry (where the forecast actually pays)
# --------------------------------------------------------------------------

def vrp_signal(iv_daily: pd.Series, pred_rv_daily: pd.Series) -> pd.Series:
    """Log richness of implied vol over the model's realized-vol forecast.

    Positive => the market is charging more for volatility than the model
    expects to be delivered => selling variance is favourably priced.
    """
    idx = iv_daily.index.intersection(pred_rv_daily.index)
    return np.log(iv_daily.loc[idx]) - np.log(pred_rv_daily.loc[idx])


def vrp_backtest(iv_daily: pd.Series, pred_rv_daily: pd.Series,
                 realized_var_fwd: pd.Series,
                 entry_z: float = 0.0, max_position: float = 1.0,
                 scale: float = 3.0,
                 allow_long_vol: bool = True,
                 hold_days: int = 5,
                 cost_vol_pts: float = 0.005,
                 risk_fraction: float = 0.15,
                 cap_multiple: float = 3.0) -> pd.DataFrame:
    """Short/long variance against the model's realized-vol forecast.

    Payoff is the standard variance-swap P&L expressed in vega terms:

        pnl_vol_points = (K^2 - RV^2) / (2K)

    with ``K`` the implied vol struck at entry and ``RV`` the volatility
    actually realized over the holding period, both annualized. ``cap_multiple``
    caps the loss at ``cap_multiple * K`` realized vol, which is what buying
    wings (a strangle overlay instead of a naked short) actually buys you --
    without it the downside is unbounded and the backtest is fiction.

    Positions are held ``hold_days`` (Nifty options expire weekly) rather than
    rebalanced daily, so the bid-ask is paid once per position, not 252 times a
    year.

    Parameters
    ----------
    realized_var_fwd
        Forward-looking: the value at ``t`` must be the *arithmetic mean daily
        variance* over ``(t, t+hold_days]`` -- ``target_rvar_h{hold_days}`` from
        features.add_targets. A variance swap settles on realized variance, not
        on the geometric mean of daily vols.
    cost_vol_pts
        Round-trip bid-ask in annualized vol points (0.005 = 0.5 vol pts).
    risk_fraction
        Position sizing, expressed as the fraction of capital a full-size
        position loses if realized vol runs all the way to the cap. 0.15 means
        the worst case on a maximal position is a 15% loss. Sharpe is invariant
        to this; the equity curve and the drawdown are not.
    """
    idx = (iv_daily.index
           .intersection(pred_rv_daily.index)
           .intersection(realized_var_fwd.index))
    iv = iv_daily.loc[idx].astype(float)
    pred = pred_rv_daily.loc[idx].astype(float)
    rvar = realized_var_fwd.loc[idx].astype(float)

    sig = np.log(iv) - np.log(pred)

    # Short variance when implied is rich, long when cheap, flat in between.
    pos = np.clip((sig - entry_z) * scale, -max_position, max_position)
    if not allow_long_vol:
        pos = pos.clip(lower=0.0)
    pos = pd.Series(pos, index=idx).fillna(0.0)

    # Enter only on rebalance dates; hold the book in between.
    active = pd.Series(0.0, index=idx)
    entry = pd.Series(False, index=idx)
    held = 0.0
    for i in range(len(idx)):
        if i % hold_days == 0:
            held = float(pos.iloc[i])
            entry.iloc[i] = True
        active.iloc[i] = held

    K = iv * np.sqrt(TRADING_DAYS)                          # annualized strike vol
    R = np.sqrt(rvar * TRADING_DAYS).clip(upper=cap_multiple * K)
    pnl_vol_pts = (K**2 - R**2) / (2 * K)                # per unit vega, short variance

    # Cost charged once per entry, on the size actually traded.
    prev = active.shift(hold_days).fillna(0.0)
    traded = (active - prev).abs().where(entry, 0.0)
    cost = traded * cost_vol_pts

    gross = active * pnl_vol_pts
    # Size so that a full position losing all the way to the cap costs
    # `risk_fraction` of capital: max loss per unit vega is
    # ((cap*K)^2 - K^2)/(2K) = K(cap^2 - 1)/2.
    max_loss_per_vega = K * (cap_multiple**2 - 1) / 2
    vega_notional = risk_fraction / max_loss_per_vega
    # A position entered at t pays over (t, t+hold_days]; spread the P&L evenly
    # so the return series is daily and Sharpe is not inflated by lumpiness.
    strat_return = (gross - cost) * vega_notional / hold_days

    out = pd.DataFrame({
        "iv_ann": K,
        "pred_rv_ann": pred * np.sqrt(TRADING_DAYS),
        "realized_rv_ann": np.sqrt(rvar * TRADING_DAYS),
        "realized_rv_ann_capped": R,
        "signal": sig,
        "position": active,
        "pnl_vol_points": gross,
        "cost": cost,
        "strategy_return": strat_return,
    })
    out["strategy_equity"] = (1 + out["strategy_return"]).cumprod()
    return out


def vrp_naive_benchmark(iv_daily: pd.Series, trailing_rv: pd.Series,
                        realized_var_fwd: pd.Series, **kwargs) -> pd.DataFrame:
    """Same trade, but sized off trailing realized vol instead of the model.

    This is the baseline the ML model has to beat *as a trading signal*. Simply
    always-short-variance harvests the premium too; the question is whether the
    forecast adds anything over that, and this is how you find out.
    """
    return vrp_backtest(iv_daily, trailing_rv, realized_var_fwd, **kwargs)


# --------------------------------------------------------------------------
# combining
# --------------------------------------------------------------------------

def combine(returns: dict[str, pd.Series],
            weights: dict[str, float] | None = None) -> pd.Series:
    """Weighted blend of strategy return streams on their shared dates.

    The vol-targeted index leg and the VRP carry leg are close to uncorrelated
    day to day -- one is long equity beta, the other is short a variance
    premium -- so blending them is the cheapest Sharpe improvement available.
    """
    df = pd.DataFrame(returns).dropna()
    if weights is None:
        weights = {k: 1.0 / len(returns) for k in returns}
    w = pd.Series(weights).reindex(df.columns).fillna(0.0)
    return (df * w).sum(axis=1)

In [ ]:
pred_vol = pd.Series(np.exp(preds["model"]) * preds["smearing"], index=preds.index)

# v1's configuration for comparison: GK units, no smearing, daily rebalance, free leverage
gk_ratio = (table["gk_vol"] / table["rv"]).reindex(preds.index).median()
pred_vol_v1 = pred_vol * gk_ratio / preds["smearing"]

bt_v1 = vol_target_backtest(pred_vol_v1, next_ret, target_ann_vol=0.16,
                             max_leverage=2.0, cost_bps=COST_BPS_CASH,
                             rf_annual=0.0, deadband=0.0)
bt_v2 = vol_target_backtest(pred_vol, next_ret, target_ann_vol=TARGET_ANN_VOL,
                             max_leverage=1.5, cost_bps=COST_BPS_CASH,
                             rf_annual=RF_ANNUAL, deadband=0.10)

print("A -- VOLATILITY OVERLAY\n")
print(pd.DataFrame([
    summarize(bt_v1["strategy_return"], label="v1 config (the bugs)"),
    summarize(bt_v2["strategy_return"], label="v2 config (fixed)"),
    summarize(bt_v2["buy_hold_return"], label="buy & hold"),
]).set_index("label")[["cagr", "ann_vol", "sharpe", "sortino", "max_drawdown", "total_return"]])

print(f"\nmean position  v1={bt_v1['position'].mean():.2f}x   v2={bt_v2['position'].mean():.2f}x")
print(f"turnover       v1={bt_v1['position'].diff().abs().sum():.0f}   v2={bt_v2['position'].diff().abs().sum():.0f}")
print(f"cost drag/yr   v1={252*bt_v1['cost'].mean():.2%}   v2={252*bt_v2['cost'].mean():.2%}")

In [ ]:
d_ls = direction["direction"].reindex(preds.index)
d_lo = build_direction(table, allow_short=False)["direction"].reindex(preds.index)

ls = long_short_backtest(d_ls, pred_vol, next_ret, target_ann_vol=TARGET_ANN_VOL,
                          max_long=1.5, max_short=-1.0, cost_bps=COST_BPS_FUT,
                          rf_annual=RF_ANNUAL, div_yield=DIV_YIELD,
                          deadband=0.10, stop_drawdown=-0.25)
lo = long_short_backtest(d_lo, pred_vol, next_ret, target_ann_vol=TARGET_ANN_VOL,
                          max_long=1.5, cost_bps=COST_BPS_FUT,
                          rf_annual=RF_ANNUAL, div_yield=DIV_YIELD,
                          deadband=0.10, stop_drawdown=-0.25)

print("B -- LONG/SHORT   (Sharpe is EXCESS of the risk-free rate)\n")
print(pd.DataFrame([
    summarize(bt_v2["strategy_return"], RF_ANNUAL, label="overlay, no direction"),
    summarize(lo["strategy_return"], RF_ANNUAL, label="directional, long-only"),
    summarize(ls["strategy_return"], RF_ANNUAL, label="directional, LONG/SHORT"),
    summarize(ls["buy_hold_return"], RF_ANNUAL, label="buy & hold (total return)"),
]).set_index("label")[["cagr", "ann_vol", "sharpe", "sortino", "max_drawdown", "total_return"]])

print(f"\ndays short {100*ls['is_short'].mean():.0f}%   "
      f"mean gross exposure {ls['gross_exposure'].mean():.2f}x   "
      f"mean net {ls['position'].mean():+.2f}x")
print("Excess-of-cash Sharpe matters here: a book that sits in cash half the time")
print("would otherwise post a flattering ratio earned on T-bills.")

In [ ]:
hold = 5
pred_rv5 = pd.Series(np.exp(preds5["model"]) * preds5["smearing"], index=preds5.index)
realized_var = table["target_rvar_h5"].reindex(pred_rv5.index)
iv = table["iv_daily"].reindex(pred_rv5.index)
trailing = np.exp(table["log_rv_w"]).reindex(pred_rv5.index)

ok = iv.notna() & pred_rv5.notna() & realized_var.notna() & trailing.notna()
iv, pred_rv5, realized_var, trailing = iv[ok], pred_rv5[ok], realized_var[ok], trailing[ok]

kw = dict(hold_days=hold, cost_vol_pts=0.005, risk_fraction=0.15, cap_multiple=3.0)
always   = vrp_backtest(iv, iv, realized_var, entry_z=-1e9, **kw)
naive_t  = vrp_backtest(iv, trailing, realized_var, **kw)
modelled = vrp_backtest(iv, pred_rv5, realized_var, **kw)

print("C -- VARIANCE RISK PREMIUM CARRY\n")
print(pd.DataFrame([
    summarize(always["strategy_return"],   label="always short variance (no timing)"),
    summarize(naive_t["strategy_return"],  label="timed off trailing realized vol"),
    summarize(modelled["strategy_return"], label="timed off the model"),
]).set_index("label")[["cagr", "ann_vol", "sharpe", "sortino", "max_drawdown", "skew", "worst_day"]])

print(f"\nmean IV/realized ratio: {(modelled['iv_ann']/modelled['realized_rv_ann']).mean():.3f}")
print(f"periods where realized exceeded implied: {(modelled['realized_rv_ann']>modelled['iv_ann']).mean():.1%}")
print("\nThe model must beat BOTH rows above it. The premium itself is free to")
print("anyone willing to be short vol; timing it is the only claim being made.")

Short variance is a negatively-skewed carry trade: many small gains, rare very
large losses. Check the `skew` and `worst_day` columns, and check whether your
sample spans **March 2020** — a Sharpe computed over a period with no volatility
crisis in it is close to meaningless for this strategy. The `cap_multiple`
assumes you buy wings (a strangle, not a naked straddle); those wings cost real
premium that this backtest only approximates.

In [ ]:
crisis = slice("2020-02-01", "2020-05-31")
if modelled.loc[crisis].shape[0] > 10:
    print("MARCH 2020 STRESS TEST\n")
    print(pd.DataFrame([
        summarize(modelled["strategy_return"].loc[crisis], label="VRP leg"),
        summarize(ls["strategy_return"].loc[crisis],       label="long/short"),
        summarize(bt_v2["strategy_return"].loc[crisis],    label="vol overlay"),
        summarize(bt_v2["buy_hold_return"].loc[crisis],    label="buy & hold"),
    ]).set_index("label")[["total_return", "max_drawdown", "worst_day"]])
else:
    print("Sample does not span the 2020 crisis -- raise PRICE_YEARS before trusting the VRP leg.")

## 11. Blend, and save everything

In [ ]:
blended = combine({"ls": ls["strategy_return"], "vrp": modelled["strategy_return"]},
                  {"ls": 0.6, "vrp": 0.4})
corr = pd.DataFrame({"a": ls["strategy_return"],
                     "b": modelled["strategy_return"]}).dropna().corr().iloc[0, 1]

results = pd.DataFrame([
    summarize(bt_v1["strategy_return"], RF_ANNUAL,  label="A. overlay, v1 config"),
    summarize(bt_v2["strategy_return"], RF_ANNUAL,  label="A. overlay, v2 config"),
    summarize(lo["strategy_return"], RF_ANNUAL,     label="B. directional long-only"),
    summarize(ls["strategy_return"], RF_ANNUAL,     label="B. directional long/short"),
    summarize(modelled["strategy_return"],          label="C. VRP carry"),
    summarize(blended,                              label="60/40 blend of B and C"),
    summarize(ls["buy_hold_return"], RF_ANNUAL,     label="buy & hold"),
]).set_index("label")

print(results[["cagr", "ann_vol", "sharpe", "sortino", "max_drawdown", "total_return"]])
print(f"\ncorrelation between the long/short and VRP legs: {corr:+.3f}")

results.to_csv(WORK / "results_summary.csv")
preds.to_csv(WORK / "walk_forward_predictions.csv")
print(f"\nsaved to {WORK}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

for name, s in [("A. overlay (v1 bugs)", bt_v1), ("A. overlay (fixed)", bt_v2),
                ("B. long/short", ls)]:
    axes[0].plot(s["strategy_equity"], lw=1.3, label=name)
axes[0].plot(bt_v2["buy_hold_equity"], lw=1.3, ls="--", color="k", label="buy & hold")
axes[0].set_title("Index strategies, net of costs, financing and carry")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(modelled["strategy_equity"], lw=1.3, label="C. VRP, model-timed")
axes[1].plot(always["strategy_equity"], lw=1.1, alpha=0.8, label="C. always short variance")
axes[1].set_title("Variance risk premium carry")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 12. What you can and cannot claim from this

**Claimable:**

- forecast error against **HAR-RV** with a Diebold-Mariano p-value — not against
  a single-day persistence strawman;
- the volatility overlay as a **risk overlay**: drawdown reduction at comparable
  return, financing charged, turnover controlled;
- the long/short book **only if** the IC table in section 9 shows a t-stat above
  2. Otherwise its Sharpe is a draw from noise and you should say so;
- the VRP strategy against **both** the untimed and the trailing-vol-timed
  benchmarks;
- sentiment's contribution as an ablation with a p-value, whichever way it lands.

**Not claimable:**

- that volatility targeting beats buy-and-hold on return — section 10A explains
  why it structurally cannot;
- a short-variance Sharpe from a sample with no volatility crisis in it;
- a long/short result whose underlying signal is not statistically significant.

**Not modelled here:** slippage beyond a fixed bps assumption, futures roll and
basis risk, margin calls, the real bid-ask on Nifty option strikes, and the cost
of the wings the `cap_multiple` stands in for. Each of these moves the answer in
the same direction: worse.